In [15]:
from __future__ import annotations

import math
import os
import sys
import time
from pathlib import Path
from typing import Tuple, Union

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from catboost import CatBoostClassifier
from classifier_calibration.calibration_error import classwise_ece
from dirichletcal.calib.fulldirichlet import FullDirichletCalibrator

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import  log_loss
from sklearn.metrics import precision_score, recall_score

In [16]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [17]:
np.random.seed(42)

In [18]:
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Thesis code" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from Functions.data_utils import (
    plot_incremental_response_rate,
    uplift_by_decile_bin,
    coerce_metrics_to_numeric,
)

In [19]:
file_path = r"Data/df_preds.csv"
df_preds = pd.read_csv(file_path)


In [22]:
df_preds['y_true'].value_counts(normalize=True)

y_true
no_reactivated_0    0.491382
no_reactivated_5    0.071349
no_reactivated_2    0.071140
no_reactivated_4    0.069462
no_reactivated_6    0.069414
no_reactivated_1    0.068732
no_reactivated_3    0.068580
no_reactivated_7    0.068030
reactivated_0       0.010313
reactivated_6       0.001797
reactivated_3       0.001735
reactivated_5       0.001726
reactivated_4       0.001674
reactivated_2       0.001598
reactivated_7       0.001550
reactivated_1       0.001517
Name: proportion, dtype: float64

## Diagnostics calibration and performance per class

In [6]:
def metrics_per_model_per_class(df: pd.DataFrame, eps: float = 1e-15) -> pd.DataFrame:
    labels = (
        [f"reactivated_{i}" for i in range(8)] +
        [f"no_reactivated_{i}" for i in range(8)]
    )
    
    rows = []
    for model, group in df.groupby("model"):
        prec = precision_score(group["y_true"], group["y_pred"], labels=labels, average=None, zero_division=0)
        rec = recall_score(group["y_true"], group["y_pred"], labels=labels, average=None, zero_division=0)
        
        for i, label in enumerate(labels):
            prob_col = f"p_{label}"
            if prob_col in group.columns:
                mean_prob = group[prob_col].mean()
                std_prob = group[prob_col].std()
                
                # OvR log loss: binary log loss of class k vs rest
                y_true_bin = (group["y_true"] == label).astype(int)
                p_k = np.clip(group[prob_col].to_numpy(), eps, 1 - eps)
                class_logloss = float(
                    log_loss(
                        y_true_bin,
                        np.column_stack([1 - p_k, p_k]),
                        labels=[0, 1],
                    )
                )
            else:
                mean_prob = np.nan
                std_prob = np.nan
                class_logloss = np.nan
            
            rows.append({
                "model": model,
                "class": label,
                "precision": prec[i],
                "recall": rec[i],
                "mean_predicted_prob": mean_prob,
                "std_predicted_prob": std_prob,
                "class_logloss": class_logloss,
            })
    
    return pd.DataFrame(rows)
    
out_dir = Path("Output/classification_output")
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df = metrics_per_model_per_class(df_preds)
metrics_df.to_excel(out_dir / "metrics_per_model_per_class.xlsx", index=False)

In [7]:
# Classwise-ECE

proba_cols = [
    "p_reactivated_0",
    "p_reactivated_1",
    "p_reactivated_2",
    "p_reactivated_3",
    "p_reactivated_4",
    "p_reactivated_5",
    "p_reactivated_6",
    "p_reactivated_7",
    "p_no_reactivated_0",
    "p_no_reactivated_1",
    "p_no_reactivated_2",
    "p_no_reactivated_3",
    "p_no_reactivated_4",
    "p_no_reactivated_5",
    "p_no_reactivated_6",
    "p_no_reactivated_7"
]

class_mapping = {
    "reactivated_0": 0,
    "reactivated_1": 1,
    "reactivated_2": 2,
    "reactivated_3": 3,
    "reactivated_4": 4,
    "reactivated_5": 5,
    "reactivated_6": 6,
    "reactivated_7": 7,
    "no_reactivated_0": 8,
    "no_reactivated_1": 9,
    "no_reactivated_2": 10,
    "no_reactivated_3": 11,
    "no_reactivated_4": 12,
    "no_reactivated_5": 13,
    "no_reactivated_6": 14,
    "no_reactivated_7": 15,
}
def compute_classwise_ece_per_model(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for model, g in df.groupby("model"):
        y_prob = g[proba_cols].to_numpy()
        y_true_int = g["y_true"].map(class_mapping).to_numpy()

        if np.isnan(y_true_int).any():
            bad_labels = (
                g.loc[np.isnan(y_true_int), "y_true"]
                .unique()
                .tolist()
            )
            raise ValueError(f"Unmapped y_true labels found for model {model}: {bad_labels}")

        y_true_int = y_true_int.astype(int)

        ece_per_class = classwise_ece(
            labels=y_true_int,
            pred_probs=y_prob
        )

        rows.append(
            {
                "model": model,
                "classwise_ece": float(np.mean(ece_per_class)),
            }
        )

    return pd.DataFrame(rows)


ece_df = compute_classwise_ece_per_model(df_preds)
ece_df

,model,classwise_ece
0,catboost_calibrated_dirichlet,0.057414
1,catboost_uncal,0.050089
2,rf_calibrated_dirichlet,0.053981
3,rf_uncal,0.025781


In [8]:
# How often did the model predict minority class and how well did it do?
for model_name in ["catboost_uncal", "catboost_calibrated_dirichlet", "rf_uncal", "rf_calibrated_dirichlet"]:
    subset = df_preds[df_preds["model"] == model_name]
    argmax_cols = subset[proba_cols].idxmax(axis=1)
    print(f"\n=== {model_name} ===")
    for col in proba_cols:
        if "reactivated_" in col and "no_" not in col:
            mask = argmax_cols == col
            count = mask.sum()
            correct = subset.loc[mask, "y_true"].eq(col.replace("p_", "")).sum()
            print(f"{col}: {count} argmax ({count/len(subset)*100:.4f}%) | {correct} correct")


=== catboost_uncal ===
p_reactivated_0: 29556 argmax (14.0138%) | 580 correct
p_reactivated_1: 4973 argmax (2.3579%) | 8 correct
p_reactivated_2: 4907 argmax (2.3266%) | 14 correct
p_reactivated_3: 6338 argmax (3.0051%) | 13 correct
p_reactivated_4: 6299 argmax (2.9866%) | 20 correct
p_reactivated_5: 6045 argmax (2.8662%) | 9 correct
p_reactivated_6: 6433 argmax (3.0502%) | 20 correct
p_reactivated_7: 4955 argmax (2.3494%) | 8 correct

=== catboost_calibrated_dirichlet ===
p_reactivated_0: 4205 argmax (1.9938%) | 62 correct
p_reactivated_1: 16258 argmax (7.7086%) | 28 correct
p_reactivated_2: 18401 argmax (8.7247%) | 47 correct
p_reactivated_3: 17285 argmax (8.1956%) | 46 correct
p_reactivated_4: 16324 argmax (7.7399%) | 36 correct
p_reactivated_5: 16611 argmax (7.8760%) | 32 correct
p_reactivated_6: 19602 argmax (9.2941%) | 37 correct
p_reactivated_7: 19404 argmax (9.2003%) | 35 correct

=== rf_uncal ===
p_reactivated_0: 182 argmax (0.0863%) | 2 correct
p_reactivated_1: 24 argmax (0.

In [9]:
# Averaged log loss over all classes, making minority classes equal in weight
def logloss_per_model_class(
    df: pd.DataFrame,
    proba_cols: list[str] = proba_cols,
    class_mapping: dict[str, int] = class_mapping,
    eps: float = 1e-15,
) -> pd.DataFrame:
    class_labels = [c.replace("p_", "") for c in proba_cols]
    inv_class_mapping = {v: k for k, v in class_mapping.items()}

    rows: list[dict] = []

    for model, g in df.groupby("model"):
        y_true_int = g["y_true"].map(class_mapping).to_numpy()

        if np.isnan(y_true_int).any():
            bad_labels = (
                g.loc[np.isnan(y_true_int), "y_true"]
                .unique()
                .tolist()
            )
            raise ValueError(f"Unmapped y_true labels found for model {model}: {bad_labels}")

        y_true_int = y_true_int.astype(int)
        y_prob = g[proba_cols].to_numpy()

        model_logloss_values = []

        # Per-class metrics (one-vs-rest)
        for k in range(len(proba_cols)):
            y_true_bin = (y_true_int == k).astype(int)

            p_k = np.clip(y_prob[:, k], eps, 1 - eps)

            ll = float(
                log_loss(
                    y_true_bin,
                    np.column_stack([1 - p_k, p_k]),
                    labels=[0, 1],
                )
            )

            model_logloss_values.append(ll)

            rows.append(
                {
                    "model": model,
                    "class": inv_class_mapping[k],
                    "logloss": ll,
                }
            )

        # ---- Mean log loss row ----
        rows.append(
            {
                "model": model,
                "class": "mean",
                "logloss": float(np.mean(model_logloss_values)),
            }
        )

    return pd.DataFrame(rows)


metrics_df = logloss_per_model_class(df_preds)
metrics_df[metrics_df['class']=='mean']

,model,class,logloss
16,catboost_calibrated_dirichlet,mean,0.230645
33,catboost_uncal,mean,0.209259
50,rf_calibrated_dirichlet,mean,0.251438
67,rf_uncal,mean,0.272431


In [10]:
# Mutliclass logloss evaluates full probbaility vector of all classes, dominated by majority group 
def multiclass_logloss_per_model(
    df: pd.DataFrame,
    proba_cols: list[str] = proba_cols,
    class_mapping: dict[str, int] = class_mapping,
) -> pd.DataFrame:
    rows = []
    for model, g in df.groupby("model"):
        y_true = g["y_true"].map(class_mapping).to_numpy().astype(int)
        y_prob = g[proba_cols].to_numpy()
        ll = log_loss(y_true, y_prob)
        rows.append({"model": model, "multiclass_logloss": float(ll)})
    return pd.DataFrame(rows)

mc_ll_df = multiclass_logloss_per_model(df_preds)
mc_ll_df

,model,multiclass_logloss
0,catboost_calibrated_dirichlet,2.722859
1,catboost_uncal,2.392160
2,rf_calibrated_dirichlet,3.079509
3,rf_uncal,3.537670


In [11]:
def logloss_per_model_class(
    df: pd.DataFrame,
    proba_cols: list[str] = proba_cols,
    class_mapping: dict[str, int] = class_mapping,
    eps: float = 1e-15,
) -> pd.DataFrame:
    inv_class_mapping = {v: k for k, v in class_mapping.items()}
    rows: list[dict] = []
    for model, g in df.groupby("model"):
        y_true_int = g["y_true"].map(class_mapping).to_numpy()
        if np.isnan(y_true_int).any():
            bad_labels = g.loc[np.isnan(y_true_int), "y_true"].unique().tolist()
            raise ValueError(f"Unmapped y_true labels found for model {model}: {bad_labels}")
        y_true_int = y_true_int.astype(int)
        y_prob = g[proba_cols].to_numpy()
        for k in range(len(proba_cols)):
            y_true_bin = (y_true_int == k).astype(int)
            p_k = np.clip(y_prob[:, k], eps, 1 - eps)
            rows.append(
                {
                    "model": model,
                    "class": inv_class_mapping[k],
                    "logloss": float(
                        log_loss(
                            y_true_bin,
                            np.column_stack([1 - p_k, p_k]),
                            labels=[0, 1],
                        )
                    ),
                }
            )
    metrics_df = pd.DataFrame(rows)
    metrics_df["logloss"] = metrics_df["logloss"].round(3)
    return metrics_df

metrics_df = logloss_per_model_class(df_preds)
OUTPUT_DIR = "Output/MTUM_phase_1_output_distribution"
os.makedirs(OUTPUT_DIR, exist_ok=True)
metrics_df.to_excel(
    os.path.join(OUTPUT_DIR, "logloss_per_model_class.xlsx"),
    index=False,
)

In [12]:
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]

INCENTIVE_NAMES = {
    0: "control", 1: "10%_discount", 2: "25%_discount", 3: "5eu_voucher",
    4: "10eu_voucher", 5: "250_loyalty_pts", 6: "500_loyalty_pts", 7: "Vitamin E",
}


def _rename_proba_cols(proba_cols):
    """Rename raw proba columns."""
    rename = {}
    for col in proba_cols:
        rest = col.split("_", 1)[1]
        idx = int(rest.rsplit("_", 1)[1])
        outcome = rest.rsplit("_", 1)[0]
        rename[col] = f"p_{outcome}_{INCENTIVE_NAMES[idx]}"
    return rename


def plot_histograms_all_pcols_by_model(
    df_preds, p_cols, model_col="model",
    nbins=20, ncols=4, opacity=0.55, plot_width=420, plot_height=280,
):
    # Melt to long format
    long = (
        df_preds[[model_col, *p_cols]]
        .melt(id_vars=model_col, var_name="prob_col", value_name="p")
        .dropna(subset=["p"])
    )
    long["prob_col"] = pd.Categorical(long["prob_col"], categories=p_cols, ordered=True)

    models = sorted(long[model_col].unique())
    color_map = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(models)}
    nrows = math.ceil(len(p_cols) / ncols)

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=list(p_cols) + [""] * (nrows * ncols - len(p_cols)),
        shared_xaxes=True, shared_yaxes=False,
        horizontal_spacing=0.06, vertical_spacing=0.09,
    )

    for i, pc in enumerate(p_cols):
        r, c = divmod(i, ncols)
        for m in models:
            vals = long.loc[(long["prob_col"] == pc) & (long[model_col] == m), "p"]
            fig.add_trace(go.Histogram(
                x=vals, xbins=dict(start=0, end=1, size=1 / nbins),
                opacity=opacity, name=str(m), legendgroup=str(m),
                showlegend=(i == 0),
                marker=dict(color=color_map[m], line=dict(color="white", width=0.6)),
            ), row=r + 1, col=c + 1)

    fig.update_layout(
        title=dict(text="Predicted probability distributions by class & model",
                   font=dict(family="Georgia, serif", size=18, color="#1a1a2e"),
                   x=0.5, xanchor="center"),
        barmode="overlay", bargap=0.05,
        width=plot_width * ncols, height=plot_height * nrows + 140,
        paper_bgcolor="#f8f9fb", plot_bgcolor="#ffffff",
        margin=dict(l=70, r=30, t=90, b=90),
        font=dict(family="Arial, sans-serif", size=11, color="#444"),
        legend=dict(title_text=model_col, orientation="h",
                    yanchor="top", y=-0.04, xanchor="center", x=0.5,
                    bgcolor="rgba(255,255,255,0.85)", bordercolor="#ddd", borderwidth=1),
    )
    fig.update_xaxes(range=[0, 1], dtick=0.25, showgrid=True, gridcolor="#e8e8e8",
                     zeroline=False, linecolor="#ccc")
    fig.update_yaxes(showgrid=True, gridcolor="#e8e8e8", zeroline=False, linecolor="#ccc")

    # Axis labels on mid row/col only
    mid_row, mid_col = math.ceil(nrows / 2), math.ceil(ncols / 2)
    for rr in range(1, nrows + 1):
        fig.update_yaxes(title_text="Count" if rr == mid_row else "", row=rr, col=1)
    for cc in range(1, ncols + 1):
        fig.update_xaxes(title_text="Predicted probability" if cc == mid_col else "", row=nrows, col=cc)

    for ann in fig.layout.annotations:
        ann.update(font=dict(size=12, color="#1a1a2e", family="Georgia, serif"))

    fig.write_html(os.path.join(OUTPUT_DIR, "Probability_histogram.html"))
    return fig


# ── Rename & plot ──
proba_col_rename = _rename_proba_cols(proba_cols)
proba_cols_renamed = list(proba_col_rename.values())
df_preds_renamed = df_preds.rename(columns=proba_col_rename)

fig_hist = plot_histograms_all_pcols_by_model(
    df_preds_renamed, p_cols=proba_cols_renamed, model_col="model"
)


In [13]:
from sklearn.calibration import calibration_curve


def _calibration_traces(df_preds, agg_cols, class_label_fn, model_col,
                        n_bins, strategy, models, color_map,
                        min_marker, max_marker, show_legend):
    """Compute calibration curves + bin counts for one subplot."""
    traces = []
    for m in models:
        sub = df_preds.loc[df_preds[model_col] == m]
        y_prob = np.clip(sub[agg_cols].sum(axis=1).values, 0, 1)
        y_binary = class_label_fn(sub).values

        if y_binary.sum() == 0 or y_binary.sum() == len(y_binary):
            continue
        try:
            frac_pos, mean_pred = calibration_curve(
                y_binary, y_prob, n_bins=n_bins, strategy=strategy)
        except Exception:
            continue

        edges = np.concatenate([
            [y_prob.min()],
            (mean_pred[:-1] + mean_pred[1:]) / 2,
            [y_prob.max() + 1e-9],
        ])
        counts = np.histogram(y_prob, bins=edges)[0]
        traces.append((m, mean_pred, frac_pos, counts))

    all_c = [n for *_, bc in traces for n in bc]
    cmin, cmax = (min(all_c), max(all_c)) if all_c else (0, 1)

    scatter_traces = []
    for m, mp, fp, bc in traces:
        sizes = ([min_marker + (max_marker - min_marker) * (n - cmin) / (cmax - cmin)
                  for n in bc] if cmax > cmin else [min_marker] * len(bc))
        hover = [f"n={n:,}<br>pred={x:.4f}<br>frac_pos={y:.4f}"
                 for n, x, y in zip(bc, mp, fp)]
        scatter_traces.append(go.Scatter(
            x=mp, y=fp, mode="lines+markers",
            name=str(m), legendgroup=str(m), showlegend=show_legend,
            marker=dict(size=sizes, color=color_map[m],
                        line=dict(color="white", width=0.8)),
            line=dict(color=color_map[m], width=2),
            hovertext=hover, hoverinfo="text",
        ))
    return scatter_traces


def _style_fig(fig, nrows, ncols, plot_width, plot_height, title_text, model_col):
    """Apply shared styling."""
    fig.update_layout(
        title=dict(text=title_text,
                   font=dict(family="Georgia, serif", size=18, color="#1a1a2e"),
                   x=0.5, xanchor="center"),
        width=plot_width * ncols,
        height=plot_height * nrows + 140,
        paper_bgcolor="#f8f9fb", plot_bgcolor="#ffffff",
        margin=dict(l=70, r=30, t=90, b=90),
        font=dict(family="Arial, sans-serif", size=11, color="#444"),
        legend=dict(title_text=model_col, orientation="h",
                    yanchor="top", y=-0.04, xanchor="center", x=0.5,
                    bgcolor="rgba(255,255,255,0.85)",
                    bordercolor="#ddd", borderwidth=1),
    )
    fig.update_xaxes(range=[0, 1], dtick=0.25, showgrid=True,
                     gridcolor="#e8e8e8", zeroline=False, linecolor="#ccc")
    fig.update_yaxes(range=[0, 1], dtick=0.25, showgrid=True,
                     gridcolor="#e8e8e8", zeroline=False, linecolor="#ccc")
    mid_row = math.ceil(nrows / 2)
    mid_col = math.ceil(ncols / 2)
    for rr in range(1, nrows + 1):
        fig.update_yaxes(title_text="Fraction of positives" if rr == mid_row else "",
                         row=rr, col=1)
    for cc in range(1, ncols + 1):
        fig.update_xaxes(title_text="Mean predicted probability" if cc == mid_col else "",
                         row=nrows, col=cc)
    for ann in fig.layout.annotations:
        ann.update(font=dict(size=12, color="#1a1a2e", family="Georgia, serif"))


# ── 16-class calibration ─────────────────────────────────────────────────────

def plot_calibration_16class(
    df_preds, p_cols, model_col="model",
    n_bins=10, ncols=4, plot_width=420, plot_height=320,
    strategy="quantile", min_marker=5, max_marker=18,
    rename_map=None,
):
    models = sorted(df_preds[model_col].unique())
    color_map = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(models)}
    nrows = math.ceil(len(p_cols) / ncols)

    display = [rename_map.get(pc, pc) if rename_map else pc for pc in p_cols]
    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=display + [""] * (nrows * ncols - len(p_cols)),
        shared_xaxes=False, shared_yaxes=False,
        horizontal_spacing=0.06, vertical_spacing=0.09,
    )

    for i, pc in enumerate(p_cols):
        r, c = divmod(i, ncols)
        class_label = pc.split("_", 1)[1]

        traces = _calibration_traces(
            df_preds, agg_cols=[pc],
            class_label_fn=lambda s, cl=class_label: (s["y_true"] == cl).astype(int),
            model_col=model_col, n_bins=n_bins, strategy=strategy,
            models=models, color_map=color_map,
            min_marker=min_marker, max_marker=max_marker,
            show_legend=(i == 0),
        )
        for tr in traces:
            fig.add_trace(tr, row=r + 1, col=c + 1)

        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1], mode="lines",
            line=dict(color="#aaa", dash="dash", width=1),
            showlegend=False,
        ), row=r + 1, col=c + 1)

    _style_fig(fig, nrows, ncols, plot_width, plot_height,
               "Calibration curves by class & model", model_col)
    fig.write_html(os.path.join(OUTPUT_DIR, "Calibration_16class.html"))
    return fig


# ── 2-class calibration ──────────────────────────────────────────────────────

def plot_calibration_2class(
    df_preds, proba_cols, model_col="model",
    n_bins=10, plot_width=500, plot_height=400,
    strategy="quantile", min_marker=6, max_marker=22,
):
    comeback_cols = [c for c in proba_cols if "no_reactivated" not in c]
    no_comeback_cols = [c for c in proba_cols if "no_reactivated" in c]

    models = sorted(df_preds[model_col].unique())
    color_map = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(models)}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["P(comeback)", "P(no_comeback)"],
        horizontal_spacing=0.10,
    )

    panels = [
        (comeback_cols, "reactivated"),
        (no_comeback_cols, "no_reactivated"),
    ]
    for col_idx, (agg_cols, class_prefix) in enumerate(panels):
        traces = _calibration_traces(
            df_preds, agg_cols=agg_cols,
            class_label_fn=lambda s, cp=class_prefix: s["y_true"].str.startswith(cp).astype(int),
            model_col=model_col, n_bins=n_bins, strategy=strategy,
            models=models, color_map=color_map,
            min_marker=min_marker, max_marker=max_marker,
            show_legend=(col_idx == 0),
        )
        for tr in traces:
            fig.add_trace(tr, row=1, col=col_idx + 1)

        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1], mode="lines",
            line=dict(color="#aaa", dash="dash", width=1),
            showlegend=False,
        ), row=1, col=col_idx + 1)

    _style_fig(fig, 1, 2, plot_width, plot_height,
               "Calibration curves — aggregated 2-class view", model_col)
    fig.write_html(os.path.join(OUTPUT_DIR, "Calibration_2class.html"))
    return fig


# ── Run both ──────────────────────────────────────────────────────────────────

fig_16 = plot_calibration_16class(
    df_preds, p_cols=proba_cols, model_col="model",
    strategy="quantile", n_bins=10,
    rename_map=proba_col_rename,
)

fig_2 = plot_calibration_2class(
    df_preds, proba_cols=proba_cols, model_col="model",
    strategy="quantile", n_bins=10,
)
